# **EXPERT DATA ANALYST — A Managed Agent for End-to-End EDA**

An upgrade of the `Sonnet 5 Data Analyst` starter. This version is **dataset-agnostic** (point it at any CSV/Excel/JSON), runs a **five-stage junior-analyst EDA protocol**, produces **charts + a written report + an Excel workbook**, and then puts its own work through a **council of reviewer agents** before publishing a revised final version.

**What changed vs. the starter notebook**

| Starter | This notebook |
| --- | --- |
| Hardcoded `parental_leave.csv` + row assertion | Any tabular file; schema discovered at runtime |
| One agent, one prompt, one turn | One analyst agent, five staged turns in a persistent session |
| No quality gate | 3-reviewer council + revision loop with a decision log |
| Agent archived at cleanup (permanent) | Agent reused across runs; archive is opt-in |
| Outputs: script, JSON, XLSX | Adds PNG charts, `eda_report.md`, self-contained `eda_report.html`, ranked insights, revision log |

**Pipeline**

`upload → profile → quality audit → target selection → transformation → uni/bi/multivariate + charts → council review → revision → publish`

### **1. Install and initialise**

Only the SDK and `python-dotenv` are needed locally — pandas, matplotlib and friends run inside Anthropic's sandbox, not on your machine.

In [ ]:
%pip install -q --upgrade anthropic python-dotenv

In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

api_key = os.environ.get("ANTHROPIC_API_KEY")
assert api_key, "Set ANTHROPIC_API_KEY in your environment or a local .env file."

# Managed Agents is a beta API. The SDK sets this header automatically for
# client.beta.{agents,environments,sessions,...} calls; we keep the constant
# because Files endpoints scoped to a session need it passed explicitly.
BETA_FLAG = "managed-agents-2026-04-01"

client = Anthropic(api_key=api_key)

# Registry of everything this notebook creates, so cleanup is exhaustive.
created = {
    "agent": None,
    "reviewer_agents": [],
    "environment": None,
    "files": [],
    "sessions": [],
}

print("Anthropic client initialised.")

### **2. Configuration — the only cell you edit between runs**

`DATASET_PATH` accepts any tabular file. `BUSINESS_CONTEXT` is the single highest-leverage field in this notebook: a target column guess is far better than a column-name heuristic when the agent knows what the data is *for*. Leave `TARGET_HINT` as `None` to let the agent nominate the primary variable itself.

In [ ]:
# ---- Inputs -----------------------------------------------------------------
DATASET_PATH = Path("parental_leave.csv")   # any .csv / .xlsx / .xls / .json / .parquet / .tsv
BUSINESS_CONTEXT = (
    "Company-level parental leave policy data. The audience is an HR analytics "
    "team benchmarking leave generosity across industries and identifying gaps "
    "between maternity and paternity provision."
)
TARGET_HINT = None          # e.g. "paid_maternity_leave"; None = agent decides and justifies

# ---- Agent ------------------------------------------------------------------
AGENT_NAME = "Expert Data Analyst"
MODEL = "claude-sonnet-5"
REUSE_AGENT = True          # reuse an existing agent of the same name instead of creating a new one
ARCHIVE_AGENT_ON_CLEANUP = False   # archiving is PERMANENT and blocks new sessions — off by default

# ---- Council ----------------------------------------------------------------
RUN_COUNCIL = True          # 3 reviewer agents audit the analyst's work
RUN_REVISION = True         # feed the council's blockers/majors back for a fix pass

# ---- Runtime ----------------------------------------------------------------
TURN_TIMEOUT_S = 2400       # per-turn wall clock guard
OUTPUT_ROOT = Path("outputs")

MOUNT_PATH = f"/workspace/{DATASET_PATH.name}"
OUTPUT_ROOT.mkdir(exist_ok=True)

assert DATASET_PATH.exists(), f"Missing input file: {DATASET_PATH.resolve()}"
print(f"Dataset: {DATASET_PATH.resolve()}")
print(f"Size: {DATASET_PATH.stat().st_size / 1_048_576:.2f} MB")
print(f"Will mount at: {MOUNT_PATH}")

### **3. The system prompt — where the "junior analyst" behaviour is engineered**

The persona is not decoration. Each rule below closes a specific failure mode of LLM data analysis: silent row dropping, means without spread, correlation stated as causation, invented numbers when a step fails, and charts whose titles describe the axes instead of the finding.

Two constraints worth calling out:

- **Flat output paths.** Everything lands directly in `/mnt/session/outputs/` with a numeric prefix. Nested directories are not reliably surfaced by the session Files listing, so a `charts/` subfolder is a good way to lose your charts.
- **A fixed insight grammar.** Forcing `finding → evidence → so-what → confidence → next check` is what separates an insight from a restated summary statistic.

In [ ]:
ANALYST_SYSTEM = """
You are "Expert Data Analyst", an AI agent that works the way a strong junior data
analyst works on their best day: methodical, transparent about uncertainty, and
never louder than the evidence.

## Working style
- Show your work. Every number you report must come from code you actually ran in
  this session. If you did not compute it, do not state it.
- Narrate in plain language a non-technical stakeholder can follow, then give the
  technical detail underneath.
- State assumptions explicitly the moment you make one. Label them ASSUMPTION.
- Distinguish association from causation in every sentence. You may say "X is
  associated with Y"; you may not say "X drives Y" unless a design supports it.
- When a step fails, report the failure and the actual error text, then adapt.
  Never fabricate a result to keep the narrative tidy. A visible failure is a
  correct outcome; an invented number is not.
- You are junior, not timid: raise concerns about the data even when they are
  inconvenient, and say clearly when a question cannot be answered with this data.
- End each stage with UNCERTAIN: a short list of what you could not determine and
  what extra data or domain input would resolve it.

## Environment
- The dataset is mounted read-only inside the sandbox. Do not modify it in place.
- Use the code execution tools. Persist real analysis in re-runnable Python
  scripts, not one-off shell pipelines, so a human can reproduce every figure.
- Save EVERY deliverable directly in /mnt/session/outputs/ with a numeric stage
  prefix. Do NOT create subdirectories inside outputs.
- Read the file defensively: try utf-8 then latin-1, sniff the delimiter, and
  handle Excel/JSON/Parquet by extension. Report the encoding and delimiter used.
- Use matplotlib with the Agg backend. Use seaborn only if importing it succeeds;
  otherwise fall back to matplotlib. Never let a missing library end a stage.

## Numeric discipline
- Report counts and percentages together, always with the denominator.
- Never report a mean without a measure of spread and the non-null n.
- Flag any statistic computed on fewer than 30 non-null observations as low-n.
- Cap categorical breakdowns at the top 15 levels plus an explicit "Other" bucket,
  and say how many levels were folded in.
- Round to a sensible precision for the unit; do not print 12 decimal places.
- Prefer Spearman alongside Pearson when a variable is visibly skewed, and say
  which one you are quoting.

## Chart rules
- One idea per chart. The title states the finding, not the variable names:
  "Paid maternity leave exceeds paternity leave in every industry" beats
  "maternity_leave vs paternity_leave".
- Always label both axes with units. Always state n in the subtitle or caption.
- No pie charts beyond 5 categories, no 3D, no dual y-axes.
- Use a colourblind-safe palette. dpi=144, bbox_inches='tight'.
- Sample large scatter plots (cap ~5000 points) and say that you sampled.
- Save as PNG with the stage prefix, e.g. 05_uni_age_distribution.png.

## Insight grammar (mandatory for every insight you publish)
INSIGHT - <the finding in one sentence>
  Evidence: <the specific numbers, with n>
  So what: <the decision or action this should inform>
  Confidence: high | medium | low, because <reason tied to data quality or n>
  Next check: <the analysis or data that would confirm or break this>

## Non-negotiables
- Never drop rows or columns silently. Every removal goes in the transformation
  decision log with a reason and the count affected.
- Never impute without recording the strategy, the columns touched, and why that
  strategy suits the missingness pattern.
- Never present a model as a deliverable. Any model you fit is a screening tool
  for feature relevance, and you must say so.
"""

print(f"System prompt: {len(ANALYST_SYSTEM)} chars")

### **4. Create (or reuse) the Expert Data Analyst agent**

Agents are persistent, versioned resources. The starter notebook archived its agent at cleanup — and archiving is **permanent and irreversible**, so every run forced a fresh agent. Here the agent is looked up by name and updated in place, which keeps one stable ID and a readable version history.

In [ ]:
AGENT_CONFIG = dict(
    model=MODEL,
    system=ANALYST_SYSTEM,
    tools=[{"type": "agent_toolset_20260401"}],
    skills=[{"type": "anthropic", "skill_id": "xlsx"}],
    description="Dataset-agnostic exploratory data analysis with council review.",
)


def get_or_create_agent(name, config, reuse=True):
    """Reuse an agent of the same name (updating its config) or create a new one."""
    if reuse:
        try:
            for existing in client.beta.agents.list():
                if getattr(existing, "name", None) != name:
                    continue
                if getattr(existing, "archived_at", None):
                    continue
                try:
                    updated = client.beta.agents.update(existing.id, **config)
                    print(f"Reused agent {existing.id} (updated to v{getattr(updated, 'version', '?')})")
                    return updated
                except Exception as exc:
                    print(f"Reused agent {existing.id} but could not update config: {exc}")
                    return existing
        except Exception as exc:
            print(f"Could not list agents ({exc}); creating a new one.")

    agent = client.beta.agents.create(name=name, **config)
    print(f"Created agent {agent.id} (v{getattr(agent, 'version', '?')})")
    return agent


agent = get_or_create_agent(AGENT_NAME, AGENT_CONFIG, reuse=REUSE_AGENT)
created["agent"] = agent.id

### **5. Sandbox environment**

`networking: limited` is the right default: the agent needs no internet to analyse a local file, and a restricted egress path is one less thing to reason about. If you later want the agent to `pip install` a package that is not pre-installed, that is when you loosen this — not before.

In [ ]:
environment = client.beta.environments.create(
    name="eda-sandbox",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

created["environment"] = environment.id
print(f"Created environment: {environment.id}")

### **6. Upload the dataset**

No row-count assertion this time. The starter asserted `row_count == 1601`, which is a correctness check for exactly one file — the first thing to go when the notebook becomes general. Instead we do a light local sniff purely so *you* can eyeball that the right file went up; the agent does its own, more careful, profiling in the sandbox.

In [ ]:
def upload_file(path, label=""):
    with open(path, "rb") as fh:
        uploaded = client.beta.files.upload(file=fh)
    created["files"].append(uploaded.id)
    print(f"Uploaded {label or path.name}: {uploaded.id}")
    return uploaded


# Local sniff — informational only, never an assertion.
if DATASET_PATH.suffix.lower() in {".csv", ".tsv", ".txt"}:
    for encoding in ("utf-8", "latin-1"):
        try:
            with DATASET_PATH.open(encoding=encoding) as fh:
                header = fh.readline().rstrip("\n")
                rows = sum(1 for _ in fh)
            print(f"Local sniff [{encoding}]: ~{rows} data rows")
            print(f"Header: {header[:200]}{'...' if len(header) > 200 else ''}")
            break
        except UnicodeDecodeError:
            continue

dataset_file = upload_file(DATASET_PATH, label="dataset")

### **7. Open the analysis session**

One session for all five stages. That matters: the session keeps both the conversation and the sandbox filesystem, so stage 4 can transform the dataframe that stage 2 diagnosed, and stage 5 can read the cleaned file stage 4 wrote. Five separate sessions would mean five cold starts and no shared reasoning.

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title=f"EDA — {DATASET_PATH.name}",
    resources=[
        {"type": "file", "file_id": dataset_file.id, "mount_path": MOUNT_PATH},
    ],
)

created["sessions"].append(session.id)
analysis_session_id = session.id
print(f"Created session: {session.id}")
print(f"Trace: https://platform.claude.com/workspaces/default/sessions/{session.id}")

### **8. Turn helper**

Two details the starter glossed over, both of which bite in longer runs:

- **Stream first, then send.** The stream only delivers events that occur after it opens, so opening it inside the `with` block before `events.send` is what keeps you from receiving the first half of the run as one buffered dump.
- **Terminal states are plural.** `session.status_idle` is the happy path, but `session.status_terminated` and `session.status_error` also end a turn. Handling only `idle` means a failed run hangs until you interrupt the kernel.

The timeout is checked as events arrive, so it bounds a slow run rather than a silent one — treat it as a guard, not a hard deadline.

In [ ]:
def run_turn(session_id, prompt, label="turn", timeout_s=TURN_TIMEOUT_S, show_text=True):
    """Send one user message and stream until the session reaches a terminal state."""
    text_parts, tools_used = [], []
    status, deadline = None, time.monotonic() + timeout_s

    print(f"\n{'=' * 78}\n  {label}\n{'=' * 78}")
    started = time.monotonic()

    with client.beta.sessions.events.stream(session_id) as stream:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": prompt}]}],
        )

        for event in stream:
            etype = getattr(event, "type", None)

            if etype == "agent.message":
                for block in getattr(event, "content", []):
                    txt = getattr(block, "text", None)
                    if txt:
                        if show_text:
                            print(txt, end="", flush=True)
                        text_parts.append(txt)

            elif etype in ("agent.tool_use", "agent.custom_tool_use"):
                name = getattr(event, "name", "<tool>")
                tools_used.append(name)
                if show_text:
                    print(f"\n  [{name}]", flush=True)

            elif etype == "session.status_idle":
                status = "idle"
                break

            elif etype in ("session.status_error", "session.status_terminated"):
                status = etype.replace("session.status_", "")
                break

            if time.monotonic() > deadline:
                status = "timeout"
                break

    elapsed = time.monotonic() - started
    print(f"\n\n[{label}] status={status} elapsed={elapsed:.0f}s tools={len(tools_used)}")

    return {
        "label": label,
        "status": status,
        "elapsed_s": round(elapsed, 1),
        "text": "".join(text_parts),
        "tools_used": tools_used,
    }

### **9. The five-stage EDA protocol**

Each stage is a separate turn with an explicit output contract. Splitting the work this way is deliberate: a single mega-prompt asking for all five deliverables reliably produces a thorough stage 1 and a thin stage 5, because the model spends its attention budget in order. Staged turns also give you a natural inspection point — if stage 3 picks the wrong target, you fix it there instead of discovering it inside a finished report.

In [ ]:
def build_stage_prompts(mount_path, context, target_hint):
    hint = (
        f"The stakeholder believes the primary variable is '{target_hint}'. "
        "Validate that choice against the data and say so if you disagree."
        if target_hint
        else "No target has been suggested. Nominate one yourself, or state clearly "
             "that this dataset has no natural target and is descriptive only."
    )

    return {
        "stage_1_profile": f"""
STAGE 1 of 5 - STRUCTURE AND STATE OF THE DATA

Dataset: {mount_path}
Business context: {context}

Load the file defensively and report:
1. Shape - number of rows and number of columns, stated separately and explicitly.
2. Every column name, its dtype, non-null count, null count and null percentage.
3. Statistical summary for numeric columns (count, mean, std, min, quartiles, max)
   and separately for non-numeric columns (count, unique, top, freq).
4. Column type census: how many numeric, categorical/object, datetime, boolean.
   List the column names under each type. Say which dtypes look wrong for their
   content - a date stored as text, a number stored as text, an ID stored as int.
5. Cardinality per column, and flag: constant columns, near-constant columns,
   likely unique identifiers, and high-cardinality free text.
6. Memory footprint and a 5-row sample.

Write /mnt/session/outputs/eda_01_profile.py and run it.
Write /mnt/session/outputs/01_profile.json with the structured results.
Finish with a plain-language paragraph a non-analyst could read, then UNCERTAIN.
""",
        "stage_2_quality": f"""
STAGE 2 of 5 - CLEANLINESS AUDIT

Audit {mount_path} for every issue below. Diagnose only - change nothing yet.

1. Duplicates: fully duplicated rows (count and %), and duplicates on any
   plausible business key. Show 3 examples if any exist.
2. Missing values: count and % per column, ranked. Describe the PATTERN - is
   missingness concentrated in certain rows, correlated across columns, or
   structural (a field that only applies to some records)?
3. Column-name hygiene: spaces, mixed case, punctuation, trailing whitespace,
   near-duplicate names, ambiguous names. Propose a snake_case rename map.
4. Errors and invalid values: numbers stored as text, dates as text, mixed types
   in one column, impossible values (negative counts, out-of-range dates,
   percentages above 100), and placeholder codes such as -999, "N/A", "unknown",
   "", "null". Report how many cells each affects.
5. Outliers: IQR and z-score flags per numeric column. FLAG ONLY - removal is a
   stage 4 decision and most outliers are real.
6. Unnecessary columns: candidates for removal - constant, over 60% missing,
   exact duplicates of another column, free-text with no analytic value, or
   pure identifiers. For each, give a KEEP or DROP recommendation WITH a reason.
   Recommending a drop is not the same as dropping it.

Produce a Data Quality Scorecard out of 100 with component scores for
completeness, uniqueness, validity, consistency, and show the arithmetic.

Write /mnt/session/outputs/eda_02_quality.py, run it, and write
/mnt/session/outputs/02_data_quality.json. Finish with UNCERTAIN.
""",
        "stage_3_target": f"""
STAGE 3 of 5 - PRIMARY AND SECONDARY VARIABLES

{hint}

1. Nominate the PRIMARY variable (the target/outcome). Justify it against:
   the business context, the column's semantics, its dtype and cardinality, its
   non-null share, its variance, and whether anything in the data would even
   explain it. Give your second and third choice with one line each on why they
   lost.
2. Classify the analytical problem: regression, binary classification,
   multiclass classification, time series, or descriptive with no target.
3. Rank the SECONDARY variables by likely relevance to the primary, using a
   quick screen appropriate to each pair:
   - numeric vs numeric: Pearson and Spearman
   - numeric vs categorical: group means with an effect size (eta-squared)
   - categorical vs categorical: chi-square with Cramer's V
   Report the statistic, the n, and whether the screen is trustworthy at that n.
4. Flag LEAKAGE suspects - columns that are consequences of the primary variable
   rather than causes of it, or that would not exist at prediction time.
5. Flag IDENTIFIER and ADMINISTRATIVE columns that should never be modelled.

These screens are univariate and unadjusted. Say so, and note that a strong
screen can vanish once you control for a confounder.

Write /mnt/session/outputs/eda_03_target.py, run it, and write
/mnt/session/outputs/03_target_selection.json. Finish with UNCERTAIN.
""",
        "stage_4_transform": f"""
STAGE 4 of 5 - TRANSFORMATION

Start again from the raw file at {mount_path} so the cleaning is reproducible in
one pass. Apply only the changes justified by stages 2 and 3, and log every one.

1. Rename columns to snake_case using your stage 2 map.
2. Fix dtypes: parse dates, coerce numerics stored as text (report how many
   values failed to coerce and what you did with them), normalise booleans.
3. Standardise categorical values: trim whitespace, unify case, merge obvious
   variants of the same level. Show every merge you make.
4. Handle missing values with a per-column strategy and a stated reason tied to
   the missingness pattern from stage 2. Do not blanket-impute the whole frame.
5. Remove exact duplicate rows. Report the count.
6. Outliers: keep by default. Winsorise or cap only where the value is
   implausible rather than merely extreme, and justify each case.
7. Derive features where they clearly help: date parts, ratios, per-capita
   normalisations, sensible bins. Explain each derived column in one line.
8. Drop only the columns you recommended for DROP in stage 2.

HARD RULE: if any single step would remove more than 5% of rows, stop, report
the impact, and flag it as REQUIRES_APPROVAL instead of proceeding.

Write /mnt/session/outputs/cleaned_dataset.csv, the script
/mnt/session/outputs/eda_04_transform.py, and
/mnt/session/outputs/04_transformations.json containing a decision log with,
per action: column, action, reason, rows or cells affected, and whether it is
reversible. Report before/after shape. Finish with UNCERTAIN.
""",
        "stage_5_analysis": """
STAGE 5 of 5 - UNIVARIATE, BIVARIATE, MULTIVARIATE ANALYSIS AND REPORT

Work from /mnt/session/outputs/cleaned_dataset.csv. Follow the chart rules and
the insight grammar in your instructions exactly.

A. UNIVARIATE
   - Each numeric column: histogram with KDE or a fitted reference, plus a box
     plot; report skew, kurtosis, mean, median, std, IQR.
   - Each categorical column: frequency bar chart, top 15 plus Other; report the
     dominant level's share and whether the column is severely imbalanced.
   - Datetime columns: coverage range, gaps, and a count-over-time line.

B. BIVARIATE - primary variable against the strongest secondaries from stage 3
   - numeric vs numeric: scatter with a trend line, correlation quoted with n.
   - numeric vs categorical: box or violin by level, group means, effect size.
   - categorical vs categorical: stacked or grouped bars, plus Cramer's V.
   Every bivariate chart needs a one-line reading of what it shows.

C. MULTIVARIATE
   - Correlation matrix heatmap over numeric columns (state the method).
   - Multicollinearity: list all pairs with |r| > 0.8 and their implication.
   - One faceted or grouped view combining at least three variables.
   - PCA scree plot and a 2-component scatter if there are 4+ numeric columns,
     with the variance explained stated.
   - If a target exists, fit ONE simple baseline model purely to rank feature
     importance. Report the metric honestly, and state plainly that this is a
     screening tool and not a modelling deliverable.
   - Where a bivariate finding reverses or weakens after controlling for another
     variable, say so explicitly. That reversal is often the real insight.

D. DELIVERABLES - all flat in /mnt/session/outputs/
   - PNG per chart, named 05_<block>_<subject>.png
   - 05_insights.json: 5 to 8 ranked insights, each in the insight grammar
   - eda_report.md: a written report - context, data at a glance, quality
     summary, target rationale, transformations, findings by block with the
     charts referenced by filename, ranked insights, limitations, recommended
     next analyses
   - eda_report.html: the same report, self-contained, with every PNG embedded
     as base64 so it opens standalone in a browser
   - expert_eda_report.xlsx via the XLSX skill, with sheets: Overview,
     Data Quality, Column Profile, Target Analysis, Transformation Log, Insights
   - eda_05_analysis.py: the re-runnable script behind all of it

Rank insights by decision value to the stated audience, not by statistical
significance. Finish with UNCERTAIN and a short "what I would do next" list.
""",
    }


STAGE_PROMPTS = build_stage_prompts(MOUNT_PATH, BUSINESS_CONTEXT, TARGET_HINT)
print("Stages:", " -> ".join(STAGE_PROMPTS))

### **10. Run the pipeline**

Runs all five stages in order against the same session. To re-run a single stage after tweaking its prompt: `transcripts["stage_3_target"] = run_turn(analysis_session_id, STAGE_PROMPTS["stage_3_target"], "stage_3_target")`.

In [ ]:
transcripts = {}

for stage_name, stage_prompt in STAGE_PROMPTS.items():
    result = run_turn(analysis_session_id, stage_prompt, label=stage_name)
    transcripts[stage_name] = result

    if result["status"] != "idle":
        print(f"\nStopping: {stage_name} ended with status={result['status']}.")
        print("Inspect the session trace above, then re-run this stage individually.")
        break

print("\n--- Pipeline summary ---")
for name, r in transcripts.items():
    print(f"{name:<20} {r['status']:<10} {r['elapsed_s']:>7.0f}s  {len(r['tools_used']):>3} tool calls")

### **11. Retrieve the outputs**

There is a short indexing lag between `session.status_idle` and the session's files appearing in the Files listing, so the helper retries rather than reporting an empty run. Files are downloaded into a per-session folder to keep multiple runs from overwriting each other.

In [ ]:
def download_session_outputs(session_id, subdir, retries=4, delay=3):
    """Download every downloadable file a session produced, with indexing-lag retries."""
    target_dir = OUTPUT_ROOT / subdir
    target_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []

    for attempt in range(retries):
        try:
            listing = client.beta.files.list(scope_id=session_id, betas=[BETA_FLAG])
        except Exception as exc:
            print(f"  files.list failed: {exc}")
            time.sleep(delay)
            continue

        candidates = [f for f in listing.data if getattr(f, "downloadable", False)]
        if candidates:
            for f in candidates:
                try:
                    content = client.beta.files.download(f.id, betas=[BETA_FLAG])
                    local_path = target_dir / Path(f.filename).name
                    content.write_to_file(str(local_path))
                    downloaded.append(local_path)
                except Exception as exc:
                    print(f"  skipped {f.filename}: {exc}")
            break

        if attempt < retries - 1:
            print(f"  no files yet (attempt {attempt + 1}/{retries}), waiting {delay}s...")
            time.sleep(delay)

    print(f"Downloaded {len(downloaded)} file(s) to {target_dir}/")
    for p in sorted(downloaded):
        print(f"  {p.name:<40} {p.stat().st_size / 1024:>8.1f} KB")
    return downloaded


analyst_outputs = download_session_outputs(analysis_session_id, "analyst")

### **12. Inspect the report and charts inline**

In [ ]:
from IPython.display import Markdown, Image, display

analyst_dir = OUTPUT_ROOT / "analyst"

report_md = analyst_dir / "eda_report.md"
if report_md.exists():
    display(Markdown(report_md.read_text(encoding="utf-8")))
else:
    print("eda_report.md not found — check the stage 5 transcript.")

for png in sorted(analyst_dir.glob("*.png")):
    print(f"\n{png.name}")
    display(Image(filename=str(png)))

insights_path = analyst_dir / "05_insights.json"
if insights_path.exists():
    print("\n--- Ranked insights ---")
    print(json.dumps(json.loads(insights_path.read_text(encoding="utf-8")), indent=2)[:4000])

### **13. The council of reviewers**

Self-review has a structural weakness: the same context that produced an error usually rationalises it. So the council runs as **separate sessions with different system prompts and no memory of how the analysis was made** — they see only the artifacts, exactly as a colleague reviewing your notebook would.

Three specialists, chosen because they fail differently:

| Reviewer | Catches |
| --- | --- |
| Data Quality Auditor | silent drops, imputation that invents signal, unaddressed duplicates, dtype coercion losses |
| Statistical Methodologist | wrong test for the data type, low-n claims, unadjusted confounding, correlation stated causally, multiple-comparison drift |
| Communication Critic | charts that don't support their claim, insights that restate a statistic, missing limitations, unreadable-to-stakeholder framing |

Each returns structured JSON with severity-tagged findings. Reviewers are explicitly barred from rewriting the analysis — a reviewer who fixes things becomes a second author and stops being a check.

In [ ]:
REVIEWER_SYSTEMS = {
    "data_quality_auditor": """
You are a Data Quality Auditor reviewing another analyst's exploratory analysis.
You did not do this work and you have no stake in defending it.

Focus exclusively on data integrity: missingness handling, imputation choices,
duplicate treatment, dtype coercion and what it silently discarded, outlier
decisions, category merges that destroy meaning, row and column removals, and
whether the transformation log accounts for every change between the raw shape
and the final shape.

Rules:
- Verify against the artifacts. Quote the actual numbers you are objecting to.
- If a claim cannot be verified from the artifacts provided, mark it
  UNVERIFIABLE. Do not assume it is wrong, and do not assume it is right.
- Do not rewrite the analysis. Diagnose and prescribe; do not produce fixed code.
- Distinguish blocker (the conclusion is unsafe as published), major (materially
  weakens a finding), and minor (polish).
- Being unable to find a real problem is a legitimate outcome. Do not manufacture
  findings to look thorough.
""",
    "statistical_methodologist": """
You are a Statistical Methodologist reviewing another analyst's exploratory
analysis. You did not do this work.

Focus exclusively on inferential validity: is each statistic appropriate to the
variable types and distributions; are correlations quoted with an n large enough
to mean anything; is Pearson used on visibly non-linear or skewed data; are
effect sizes reported alongside p-values; is multiple comparison acknowledged
when many pairs were screened; are confounders considered before a bivariate
finding is promoted to an insight; is any causal language present that the design
cannot support; does the baseline model's metric get over-read.

Rules:
- Quote the specific statistic and value you are challenging.
- Mark anything you cannot verify from the artifacts as UNVERIFIABLE.
- Do not rewrite the analysis. Name the flaw and the correct approach.
- Severity: blocker, major, minor.
- If the statistics are sound, say so plainly rather than inventing objections.
""",
    "communication_critic": """
You are a Communication and Visualisation Critic reviewing another analyst's
exploratory analysis, on behalf of the stakeholder who will actually read it.

Focus exclusively on whether the work communicates: does each chart's design
support the claim it is attached to; are chart types appropriate; are axes
labelled with units and is n stated; do titles carry the finding; are insights
genuine decisions-relevant findings rather than restated summary statistics; is
confidence honestly expressed; are limitations stated where a reader could
otherwise over-read a result; could a non-technical stakeholder follow the
report end to end without the analyst present.

Rules:
- Reference specific chart filenames and specific insight text.
- Mark anything you cannot verify from the artifacts as UNVERIFIABLE.
- Do not rewrite the report. Say what is wrong and what would fix it.
- Severity: blocker, major, minor.
- Clarity praise is useful when deserved; do not pad the review with objections.
""",
}

REVIEW_TASK = """
You are reviewing an exploratory data analysis. The analyst's artifacts are
mounted under /workspace/review/. Read them before forming any opinion.

Business context given to the analyst:
{context}

Review scope reminder: stay strictly inside your specialism. Another reviewer
covers the areas outside it, and overlapping reviews dilute the signal.

Produce /mnt/session/outputs/review_{role}.json with these top-level keys:
  role             - your reviewer role, as a string
  verdict          - one of: approve, revise, reject
  scores           - an object of criterion name to integer 0-5
  findings         - an array; each element has: severity (blocker, major or
                     minor), location (file or section), issue, evidence (the
                     actual number or text you are citing), fix (what the analyst
                     should do), and confidence (high, medium, low)
  unverifiable     - an array of claims you could not check against the artifacts
  strengths        - an array of things done well, with specifics
  top_3_fixes      - an array of the three highest-value changes, in order

Also print a short readable summary of your verdict and top findings.
"""

print(f"{len(REVIEWER_SYSTEMS)} reviewers configured.")

### **14. Run the council**

Each reviewer runs as a session on the same base agent using `agent_with_overrides`, which swaps the system prompt for that session only — no extra agent resources to create or clean up. If your workspace does not accept the overrides shape, the helper falls back to creating a dedicated reviewer agent, so the notebook degrades instead of failing.

In [ ]:
REVIEW_ARTIFACTS = [
    "eda_report.md",
    "01_profile.json",
    "02_data_quality.json",
    "03_target_selection.json",
    "04_transformations.json",
    "05_insights.json",
]


def create_reviewer_session(role, system_prompt, artifact_file_ids):
    """Session-local system override, with a dedicated-agent fallback."""
    resources = [
        {"type": "file", "file_id": fid, "mount_path": f"/workspace/review/{name}"}
        for name, fid in artifact_file_ids.items()
    ]
    try:
        return client.beta.sessions.create(
            agent={"type": "agent_with_overrides", "id": agent.id, "system": system_prompt},
            environment_id=environment.id,
            title=f"Council review — {role}",
            resources=resources,
        )
    except Exception as exc:
        print(f"  overrides unavailable ({exc}); creating a dedicated reviewer agent.")
        reviewer_agent = client.beta.agents.create(
            name=f"Reviewer — {role}",
            model=MODEL,
            system=system_prompt,
            tools=[{"type": "agent_toolset_20260401"}],
        )
        created["reviewer_agents"].append(reviewer_agent.id)
        return client.beta.sessions.create(
            agent=reviewer_agent.id,
            environment_id=environment.id,
            title=f"Council review — {role}",
            resources=resources,
        )


reviews = {}

if RUN_COUNCIL:
    artifact_ids = {}
    for name in REVIEW_ARTIFACTS:
        path = analyst_dir / name
        if path.exists():
            artifact_ids[name] = upload_file(path, label=f"artifact/{name}").id
        else:
            print(f"  missing artifact, skipping: {name}")

    for png in sorted(analyst_dir.glob("*.png"))[:12]:
        artifact_ids[png.name] = upload_file(png, label=f"artifact/{png.name}").id

    for role, system_prompt in REVIEWER_SYSTEMS.items():
        rsession = create_reviewer_session(role, system_prompt, artifact_ids)
        created["sessions"].append(rsession.id)

        result = run_turn(
            rsession.id,
            REVIEW_TASK.format(context=BUSINESS_CONTEXT, role=role),
            label=f"council/{role}",
        )
        review_files = download_session_outputs(rsession.id, f"review/{role}")

        parsed = None
        for path in review_files:
            if path.suffix == ".json":
                try:
                    parsed = json.loads(path.read_text(encoding="utf-8"))
                    break
                except json.JSONDecodeError:
                    continue

        reviews[role] = {"session_id": rsession.id, "result": result, "review": parsed}
else:
    print("Council skipped (RUN_COUNCIL = False).")

### **15. Aggregate the council's verdict**

Blockers and majors become the revision brief. Minors are reported but not acted on automatically — a fix pass that chases twenty small notes tends to churn the report without improving it.

In [ ]:
def summarise_council(reviews):
    rows, brief_items = [], []

    for role, payload in reviews.items():
        review = payload.get("review")
        if not review:
            rows.append((role, "no-json", 0, 0, 0))
            continue

        findings = review.get("findings", []) or []
        by_sev = {"blocker": 0, "major": 0, "minor": 0}
        for f in findings:
            sev = str(f.get("severity", "minor")).lower()
            by_sev[sev] = by_sev.get(sev, 0) + 1
            if sev in ("blocker", "major"):
                brief_items.append((sev, role, f))

        rows.append((role, review.get("verdict", "?"),
                     by_sev["blocker"], by_sev["major"], by_sev["minor"]))

    print(f"{'reviewer':<28} {'verdict':<10} {'block':>6} {'major':>6} {'minor':>6}")
    for r in rows:
        print(f"{r[0]:<28} {r[1]:<10} {r[2]:>6} {r[3]:>6} {r[4]:>6}")

    brief_items.sort(key=lambda x: 0 if x[0] == "blocker" else 1)
    return brief_items


council_items = summarise_council(reviews) if reviews else []

print(f"\n{len(council_items)} blocker/major finding(s) for the revision brief.\n")
for sev, role, f in council_items[:10]:
    print(f"[{sev.upper()}] ({role}) {f.get('location', '?')}")
    print(f"   issue: {f.get('issue', '')}")
    print(f"   fix:   {f.get('fix', '')}\n")

### **16. Revision pass**

The brief goes back to the *original* session, so the analyst still has the full reasoning behind every choice being questioned. Note the instruction to push back with evidence where a reviewer is wrong: an analyst who accepts all feedback uncritically is as unreliable as one who accepts none, and reviewers working from artifacts alone do sometimes misread them.

In [ ]:
def build_revision_brief(items, limit=10):
    lines = [
        "REVISION PASS - COUNCIL REVIEW FEEDBACK",
        "",
        "Three independent reviewers audited your artifacts. They saw the outputs",
        "only, not your reasoning. Their blocker and major findings follow.",
        "",
    ]
    for i, (sev, role, f) in enumerate(items[:limit], 1):
        lines += [
            f"{i}. [{sev.upper()}] from {role}",
            f"   Location: {f.get('location', 'unspecified')}",
            f"   Issue: {f.get('issue', '')}",
            f"   Evidence cited: {f.get('evidence', '')}",
            f"   Suggested fix: {f.get('fix', '')}",
            "",
        ]
    lines += [
        "For EACH item: accept it and fix it, or reject it with the evidence that",
        "shows the reviewer misread the artifacts. Both are valid responses and a",
        "reasoned rejection is more valuable than a compliant rewrite of something",
        "that was already correct.",
        "",
        "Then regenerate every artifact affected by an accepted fix - the charts,",
        "05_insights.json, eda_report.md, eda_report.html and expert_eda_report.xlsx",
        "must all end up consistent with each other.",
        "",
        "Write /mnt/session/outputs/06_revision_log.md recording, per item:",
        "the finding, your decision (accepted or rejected), your reasoning, and",
        "exactly which files changed. Finish with a one-paragraph statement of what",
        "a reader should still treat with caution in the final report.",
    ]
    return "\n".join(lines)


if RUN_REVISION and council_items:
    revision_prompt = build_revision_brief(council_items)
    transcripts["revision"] = run_turn(analysis_session_id, revision_prompt, label="revision")
    final_outputs = download_session_outputs(analysis_session_id, "final")
elif RUN_REVISION:
    print("No blocker/major findings — nothing to revise. That is a valid outcome.")
else:
    print("Revision skipped (RUN_REVISION = False).")

In [ ]:
final_dir = OUTPUT_ROOT / "final"
revision_log = final_dir / "06_revision_log.md"

if revision_log.exists():
    display(Markdown(revision_log.read_text(encoding="utf-8")))

final_report = final_dir / "eda_report.md"
if final_report.exists():
    display(Markdown(final_report.read_text(encoding="utf-8")))

print("\nFinal deliverables:")
for p in sorted(final_dir.glob("*")):
    print(f"  {p.name}")

### **17. Optional: native multi-agent and outcome checks**

Two Managed Agents features would replace parts of this notebook's hand-rolled machinery:

- **Outcomes** (`user.define_outcome`) — you supply a rubric and the agent self-evaluates against it, iterating until it passes.
- **Multi-agent coordination** (`multiagent: {"type": "coordinator", ...}`) — a coordinator agent delegates to sub-agents natively, with per-sub-agent event threads.

Both are **research preview** and require access approval, so the cell below is guarded. The sequential council above is the version that works on the public beta today; treat this as the migration path, not the default.

In [ ]:
TRY_RESEARCH_PREVIEW = False   # set True only if your org has research-preview access

if TRY_RESEARCH_PREVIEW:
    rubric = (
        "PASS requires all of: every reported number traceable to executed code; "
        "no causal language unsupported by the design; every chart labelled with "
        "units and n; every row or column removal present in the transformation "
        "log; at least 5 insights in the required grammar; limitations stated."
    )
    try:
        client.beta.sessions.events.send(
            analysis_session_id,
            events=[{
                "type": "user.define_outcome",
                "description": "Publish a defensible EDA report for this dataset.",
                "rubric": {"type": "text", "content": rubric},
                "max_iterations": 3,
            }],
        )
        print("Outcome defined — the agent will self-evaluate against the rubric.")
    except Exception as exc:
        print(f"Outcomes unavailable on this workspace: {exc}")
else:
    print("Research-preview features skipped.")

### **18. Cleanup**

Sessions and environments hold billable resources; delete them. The **agent is kept** — it is the reusable asset, and archiving is irreversible. Uploaded files are deleted, but anything already downloaded to `outputs/` stays on disk.

In [ ]:
def safe(label, fn):
    try:
        fn()
        print(f"deleted {label}")
    except Exception as exc:
        print(f"could not delete {label}: {exc}")


for sid in created["sessions"]:
    safe(f"session {sid}", lambda s=sid: client.beta.sessions.delete(s))

for fid in created["files"]:
    safe(f"file {fid}", lambda f=fid: client.beta.files.delete(f))

if created["environment"]:
    safe("environment", lambda: client.beta.environments.delete(created["environment"]))

if ARCHIVE_AGENT_ON_CLEANUP:
    for aid in [created["agent"], *created["reviewer_agents"]]:
        if aid:
            safe(f"agent {aid} (archived, permanent)", lambda a=aid: client.beta.agents.archive(a))
else:
    print(f"\nAgent kept for reuse: {created['agent']}")
    if created["reviewer_agents"]:
        print(f"Reviewer agents kept: {created['reviewer_agents']}")

print("\nCleanup complete. Outputs remain in ./outputs/")

### **19. Running it on a different dataset**

Change `DATASET_PATH`, `BUSINESS_CONTEXT` and optionally `TARGET_HINT` in the config cell, then run from section 5 down. Nothing else is dataset-specific.

**Extensions worth building next**

- **Prompt-level:** add a stage 0 that asks the agent to state what it expects to find *before* it looks, then score its own priors in stage 5. It is a cheap and surprisingly effective check on post-hoc storytelling.
- **Council:** add a Domain Expert reviewer whose system prompt carries your industry's rules — the checks that catch errors no general-purpose reviewer can see.
- **Governance:** persist `04_transformations.json` and `06_revision_log.md` per dataset version. Between them they are a complete, auditable record of what happened to the data and why.
- **Scale:** wrap sections 5–16 in a function and drive it from a folder of datasets, or move it to a scheduled deployment for recurring reports.